# AI-suggested HEp-2 IIF interpretation and specific autoantibodies

This notebook reproduces the analyses reported in the manuscript **from the final merged analytic dataset onward**.  
Raw HEp-2 IIF/autoantibody assay extraction, cleaning, deduplication, and linkage are intentionally outside the scope of this public notebook.

The notebook is organized in the same order as the manuscript Results:

1. **HEp-2 IIF pattern and autoantibody characteristics** → Table 1  
2. **Changes in AI-suggested HEp-2 IIF interpretations following human review** → Figure 2  
3. **HEp-2 IIF agreement with autoantibody positivity and pattern–autoantibody associations** → Figure 3 and Supplementary Table S2  
4. **AI probability distributions and discrimination for autoantibody positivity** → Figure 4, Table 2, and Supplementary Table S3  
5. **AI-guided anti-DFS70 testing strategy** → Figure 5A  

### Analysis conventions

- Autoantibody positivity for the primary analyses is defined as **≥2+**, corresponding to band intensity **≥26**.
- Manufacturer-defined AI pattern positivity is **probability ≥50%**. In the source export, probability values below the reporting threshold may be blank.
- For binary AI-pattern analyses, a blank probability is therefore treated as the pattern not being suggested.
- For continuous probability analyses (median/IQR, Mann–Whitney U test, and c-statistic), only cases with an available AI probability are used.
- Autoantibody-positive **case counts are case-level**: one patient with multiple positive autoantibodies is counted once. Autoantibody profile counts are antibody-level and are therefore not mutually exclusive.


## 0. Setup

In [ ]:
from pathlib import Path
from math import exp, log, sqrt
import sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.metrics import auc, confusion_matrix, roc_curve
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

RANDOM_SEED = 42

# Paths
DATA_PATH = Path("merged_ana_lia.csv")

# Analysis thresholds
AB_POS_THRESHOLD = 26       # >=2+ by line immunoassay
AI_PATTERN_THRESHOLD = 50   # manufacturer-defined pattern positivity
DFS_RULEIN_THRESHOLD = 85


# Reproducible plotting defaults
sns.set_theme(style="white")
mpl.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "savefig.dpi": 600,
})

MAIN_GREEN = "#48967E"
DARK_GREEN = "#1F6049"
LIGHT_GREEN = "#B3DBCF"
cmap_green = sns.light_palette(MAIN_GREEN, as_cmap=True)

In [ ]:
# Central variable definitions

AI_PATTERNS = {
    "Homogeneous": "AC_1_P_AI",
    "Dense fine speckled": "AC_2_P_AI",
    "Centromere": "AC_3_P_AI",
    "Nuclear speckled": "SPECKLED_P_AI",
    "Nuclear dot": "DOTS_P_AI",
    "Nucleolar": "NUCLEOLAERE_P_AI",
    "Cytoplasmic": "CYTO_P_AI",
    "Negative": "NEG_P_AI",
}

HUMAN_PATTERNS = {
    "Homogeneous": "NUC__AC_1_M",
    "Centromere": "NUC__AC_3_M",
    "Nuclear speckled": "NUC__AC_2_4_5_29_M",
    "Nuclear dot": "NUC__AC_6_7_M",
    "Nucleolar": "NUC__AC_8_9_10_M",
    "Nuclear envelope": "NUC__AC_11_12_M",
    "Nuclear pleomorphic": "NUC__AC_13_14_M",
    "Cytoplasmic": "CYT_M",
    "Mitotic": "MIT_M",
    "Negative": "NEG_M",
}

# nuclear pattern pairs used in the manuscript analysis.
TITER_PATTERN_PAIRS = [
    ("AC_1_P_AI", "NUC__AC_1_M"),
    ("AC_3_P_AI", "NUC__AC_3_M"),
    ("SPECKLED_P_AI", "NUC__AC_2_4_5_29_M"),
    ("DOTS_P_AI", "NUC__AC_6_7_M"),
    ("NUCLEOLAERE_P_AI", "NUC__AC_8_9_10_M"),
]

AUTOANTIBODIES = [
    "dsDNA", "HI", "NUC", "DFS70",
    "CB", "CA",
    "SSA", "Ro-52", "SSB",
    "Ku", "Mi-2b", "Mi-2a", "Sm", "RNP/Sm",
    "RP155", "RP11", "Sp100", "PML",
    "PM75", "PM100", "Scl-70", "gp210", "PCNA",
]

AI_AB_MAP = {
    "Homogeneous": {
        "ai_col": "AC_1_P_AI",
        "antibodies": ["dsDNA", "HI", "NUC", "Scl-70"],
    },
    "Dense fine speckled": {
        "ai_col": "AC_2_P_AI",
        "antibodies": ["DFS70"],
    },
    "Centromere": {
        "ai_col": "AC_3_P_AI",
        "antibodies": ["CB", "CA"],
    },
    "Nuclear speckled": {
        "ai_col": "SPECKLED_P_AI",
        "antibodies": [
            "SSA", "Ro-52", "SSB", "Ku",
            "Mi-2b", "Mi-2a", "Sm", "RNP/Sm", "RP155", "RP11",
        ],
    },
    "Nuclear dots": {
        "ai_col": "DOTS_P_AI",
        "antibodies": ["Sp100", "PML"],
    },
    "Nucleolar": {
        "ai_col": "NUCLEOLAERE_P_AI",
        "antibodies": ["PM75", "PM100", "Scl-70"],
    },
}

#Antibody display
AB_DISPLAY = {
    "dsDNA": "Anti-dsDNA",
    "HI": "Anti-Histone",
    "NUC": "Anti-Nucleosome",
    "DFS70": "Anti-DFS70",
    "CB": "Anti-CENP B",
    "CA": "Anti-CENP A",
    "SSA": "Anti-TROVE2/Ro60",
    "Ro-52": "Anti-TRIM21/Ro52",
    "SSB": "Anti-SSB/La",
    "Ku": "Anti-Ku",
    "Mi-2b": "Anti-Mi2b",
    "Mi-2a": "Anti-Mi2a",
    "Sm": "Anti-Sm",
    "RNP/Sm": "Anti-RNP/Sm",
    "RP155": "Anti-RP155",
    "RP11": "Anti-RP11",
    "Sp100": "Anti-SP100",
    "PML": "Anti-PML",
    "PM75": "Anti-PM/Scl-75",
    "PM100": "Anti-PM/Scl-100",
    "Scl-70": "Anti-Scl70",
    "gp210": "Anti-GP210",
    "PCNA": "Anti-PCNA",
}

# interpretations shown in the manuscript figure.
FIG3_PATTERN_MAP = {
    "Homogeneous (AI)": "AC_1_P_AI",
    "Homogeneous (Human)": "NUC__AC_1_M",
    "Dense fine speckled (AI)": "AC_2_P_AI",
    "Centromere (AI)": "AC_3_P_AI",
    "Centromere (Human)": "NUC__AC_3_M",
    "Nuclear speckled (AI)": "SPECKLED_P_AI",
    "Nuclear speckled (Human)": "NUC__AC_2_4_5_29_M",
    "Nuclear dots (AI)": "DOTS_P_AI",
    "Nuclear dots (Human)": "NUC__AC_6_7_M",
    "Nucleolar (AI)": "NUCLEOLAERE_P_AI",
    "Nucleolar (Human)": "NUC__AC_8_9_10_M",
    "Nuclear envelope (Human)": "NUC__AC_11_12_M",
    "Nuclear pleomorphic (Human)": "NUC__AC_13_14_M",
    "Negative (AI)": "NEG_P_AI",
    "Negative (Human)": "NEG_M",
}

In [ ]:
# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------

def load_merged_data(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError("DATA_PATH must be .csv, .xlsx/.xls, or .parquet")


def numeric_series(series):
    return pd.to_numeric(series, errors="coerce")


def ai_positive(series, threshold=AI_PATTERN_THRESHOLD):
    return numeric_series(series).fillna(0).ge(threshold)


def human_positive(series):
    return numeric_series(series).fillna(0).gt(0)


def collapse_titer(value):
    if pd.isna(value):
        return np.nan
    value = float(value)
    if value <= 100:
        return "1:100"
    if value <= 320:
        return "1:320"
    if value <= 1000:
        return "1:1000"
    return ">1:1000"


def wilson_ci(success, total):
    if total == 0:
        return np.nan, np.nan
    return proportion_confint(success, total, alpha=0.05, method="wilson")


def bootstrap_auc_ci(y_true, y_score, n_bootstrap=2000, seed=RANDOM_SEED):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_score = pd.Series(y_score).reset_index(drop=True)
    rng = np.random.default_rng(seed)
    aucs = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(y_true), len(y_true))
        y_b = y_true.iloc[idx]
        s_b = y_score.iloc[idx]
        if y_b.nunique() < 2:
            continue
        fpr_b, tpr_b, _ = roc_curve(y_b, s_b)
        aucs.append(auc(fpr_b, tpr_b))

    if not aucs:
        return np.nan, np.nan
    return tuple(np.percentile(aucs, [2.5, 97.5]))

## 1. Load the final merged analytic dataset

In [ ]:
df = load_merged_data(DATA_PATH).copy()

required_columns = set(AI_PATTERNS.values()) | set(HUMAN_PATTERNS.values()) | set(AUTOANTIBODIES)

# AI titer variables needed for Figure 2B
for ai_prob_col, _ in TITER_PATTERN_PAIRS:
    required_columns.add(ai_prob_col.replace("_P_AI", "_T_AI"))

missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise KeyError(
        "The merged dataset is missing required columns:\n- "
        + "\n- ".join(missing_columns)
    )

# Numeric conversion only for analysis variables.
analysis_numeric_columns = sorted(required_columns)
df[analysis_numeric_columns] = df[analysis_numeric_columns].apply(pd.to_numeric, errors="coerce")

N = len(df)

print(f"Analytic cases: {N:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())

# Result 1. HEp-2 IIF pattern and autoantibody characteristics

This section reproduces the descriptive counts used in **Table 1**.
Pattern counts are not mutually exclusive because a case may contain more than one HEp-2 IIF pattern.
Specific autoantibody positivity is defined as band intensity ≥26 (≥2+).

In [ ]:
# Table 1A: AI-suggested pattern distribution
ai_table_rows = []
for label, col in AI_PATTERNS.items():
    n = int(ai_positive(df[col]).sum())
    ai_table_rows.append({
        "AI-suggested pattern": label,
        "n": n,
        "%": round(n / N * 100, 1),
    })

table1_ai = pd.DataFrame(ai_table_rows)

# Table 1B: Human-reviewed pattern distribution
human_table_rows = []
for label, col in HUMAN_PATTERNS.items():
    n = int(human_positive(df[col]).sum())
    human_table_rows.append({
        "Human-reviewed pattern": label,
        "n": n,
        "%": round(n / N * 100, 1),
    })

table1_human = pd.DataFrame(human_table_rows)


# Table 1C: Specific autoantibody distribution
ab_table_rows = []
for ab in AUTOANTIBODIES:
    n = int(df[ab].ge(AB_POS_THRESHOLD).sum())
    ab_table_rows.append({
        "Autoantibody": AB_DISPLAY.get(ab, ab),
        "n": n,
        "%": round(n / N * 100, 1),
    })

any_ab_positive = df[AUTOANTIBODIES].ge(AB_POS_THRESHOLD).any(axis=1)
ab_negative_n = int((~any_ab_positive).sum())

table1_ab = pd.DataFrame(ab_table_rows)
table1_ab.loc[len(table1_ab)] = {
    "Autoantibody": "Negative",
    "n": ab_negative_n,
    "%": round(ab_negative_n / N * 100, 1),
}

display(table1_ai)
display(table1_human)
display(table1_ab)

# Result 2. Concordance between AI-suggested and human-reviewd HEp-2 IIF interpretations

**Figure 2A** shows concordance and changes in AI-suggested HEp-2 IIF patterns following human review.  
**Figure 2B** shows changes in titer among pattern-concordant nuclear results.

In [ ]:
# Figure 2A: AI-suggested → human-reviewed pattern reassignment
ai_binary = pd.DataFrame({
    label: ai_positive(df[col])
    for label, col in AI_PATTERNS.items()
})

human_binary = pd.DataFrame({
    label: human_positive(df[col])
    for label, col in HUMAN_PATTERNS.items()
})

pattern_matrix = pd.DataFrame(
    index=list(AI_PATTERNS),
    columns=list(HUMAN_PATTERNS),
    dtype=float,
)
pattern_annot = pd.DataFrame(
    "", index=list(AI_PATTERNS), columns=list(HUMAN_PATTERNS), dtype=object
)

for ai_label in AI_PATTERNS:
    ai_mask = ai_binary[ai_label]
    denom = int(ai_mask.sum())

    for human_label in HUMAN_PATTERNS:
        overlap = int((ai_mask & human_binary[human_label]).sum())

        if denom == 0:
            pattern_matrix.loc[ai_label, human_label] = np.nan
            continue

        pct = overlap / denom * 100
        pattern_matrix.loc[ai_label, human_label] = pct
        pattern_annot.loc[ai_label, human_label] = (
            f"{overlap} / {denom}\n({pct:.1f}%)"
        )

display(pattern_matrix)

In [ ]:
# Figure 2B: Titer reassignment
transition_rows = []

for ai_col, human_col in TITER_PATTERN_PAIRS:
    ai_titer_col = ai_col.replace("_P_AI", "_T_AI")

    mask = (
        ai_positive(df[ai_col])
        & human_positive(df[human_col])
        & df[ai_titer_col].notna()
        & df[human_col].notna()
    )

    temp = df.loc[mask, [ai_titer_col, human_col]].copy()
    temp.columns = ["AI_titer", "Human_titer"]
    transition_rows.append(temp)

transition_df = pd.concat(transition_rows, ignore_index=True)

transition_df["AI_group"] = transition_df["AI_titer"].apply(collapse_titer)
transition_df["Human_group"] = transition_df["Human_titer"].apply(collapse_titer)

TITER_ORDER = ["1:100", "1:320", "1:1000", ">1:1000"]

titer_count = (
    pd.crosstab(transition_df["AI_group"], transition_df["Human_group"])
    .reindex(index=TITER_ORDER, columns=TITER_ORDER, fill_value=0)
)

n_titer_pairs = len(transition_df)
titer_matrix = (titer_count / n_titer_pairs * 100).round(1)
titer_annot = (titer_matrix.astype(str) + "%").mask(titer_count.eq(0), "")

rank = {label: i for i, label in enumerate(TITER_ORDER, start=1)}
ai_rank = transition_df["AI_group"].map(rank)
human_rank = transition_df["Human_group"].map(rank)

titer_direction = pd.DataFrame({
    "Direction": ["Increase", "Decrease", "No change"],
    "n": [
        int((human_rank > ai_rank).sum()),
        int((human_rank < ai_rank).sum()),
        int((human_rank == ai_rank).sum()),
    ],
})
titer_direction["%"] = (titer_direction["n"] / n_titer_pairs * 100).round(1)

print(f"Matched pattern–titer pairs: {n_titer_pairs}")
display(titer_direction)

In [ ]:
# Figure 2: combined figure

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(11, 13),
    gridspec_kw={"height_ratios": [8, 4], "hspace": 0.55},
)

sns.heatmap(
    pattern_matrix,
    cmap=cmap_green,
    vmin=0, vmax=100,
    annot=pattern_annot, fmt="",
    linewidths=0.5, linecolor="white",
    cbar=False,
    annot_kws={"size": 8, "color": "black"},
    ax=ax1,
)
ax1.set_title("A.", loc="left", fontweight="bold", fontsize=15)
ax1.set_xlabel("Human-reviewed pattern", fontweight="bold")
ax1.set_ylabel("AI-suggested pattern", fontweight="bold")
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=35, ha="right")
ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0)

sns.heatmap(
    titer_matrix,
    cmap=cmap_green,
    vmin=0, vmax=100,
    annot=titer_annot, fmt="",
    linewidths=0.5, linecolor="white",
    cbar=False,
    annot_kws={"size": 11, "color": "black"},
    ax=ax2,
)
ax2.set_title("B.", loc="left", fontweight="bold", fontsize=15)
ax2.set_xlabel("Human-reviewed titer", fontweight="bold")
ax2.set_ylabel("AI-suggested titer", fontweight="bold")
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=35, ha="right")
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0)

plt.show()

# Result 3. HEp-2 IIF agreement with autoantibody positivity and pattern–autoantibody associations

**Figure 3** shows pattern–autoantibody associations and the positive percentage agreement (PPA) of AI-suggested and human-reviewed HEp-2 IIF patterns with corresponding autoantibody positivity. PPA is calculated within each autoantibody-positive group as the proportion exhibiting the indicated HEp-2 IIF pattern.  
**Supplementary Table S2** describes the autoantibody profiles of positive-to-negative and negative-to-positive reclassifications following human review.

In [ ]:
# Overall agreement with autoantibody positivity

# Autoantibody positivity (>=2+)
ab_positive = df[AUTOANTIBODIES].ge(AB_POS_THRESHOLD).any(axis=1)

# Overall HEp-2 IIF positivity
ai_iif_positive = pd.DataFrame({
    label: ai_positive(df[col])
    for label, col in AI_PATTERNS.items()
    if label != "Negative"
}).any(axis=1)

human_iif_positive = pd.DataFrame({
    label: human_positive(df[col])
    for label, col in HUMAN_PATTERNS.items()
    if label != "Negative"
}).any(axis=1)


# ------------------------------------------------------------
# PPA and NPA
# ------------------------------------------------------------

agreement_rows = []

for method, iif_positive in {
    "AI-suggested": ai_iif_positive,
    "Human-reviewed": human_iif_positive,
}.items():

    tp = int((ab_positive & iif_positive).sum())
    fn = int((ab_positive & ~iif_positive).sum())
    tn = int((~ab_positive & ~iif_positive).sum())
    fp = int((~ab_positive & iif_positive).sum())

    ppa = tp / (tp + fn)
    npa = tn / (tn + fp)

    ppa_low, ppa_high = wilson_ci(tp, tp + fn)
    npa_low, npa_high = wilson_ci(tn, tn + fp)

    agreement_rows.append({
        "Interpretation": method,
        "PPA (%)": round(ppa * 100, 1),
        "PPA 95% CI": f"{ppa_low*100:.1f}–{ppa_high*100:.1f}",
        "NPA (%)": round(npa * 100, 1),
        "NPA 95% CI": f"{npa_low*100:.1f}–{npa_high*100:.1f}",
    })

overall_agreement = pd.DataFrame(agreement_rows)

display(overall_agreement)


# ------------------------------------------------------------
# Paired comparison: AI vs human review (McNemar)
# ------------------------------------------------------------

def compare_paired(ai_result, human_result, reference_status):

    ai_correct = ai_result.eq(reference_status)
    human_correct = human_result.eq(reference_status)

    table = pd.crosstab(ai_correct, human_correct).reindex(
        index=[False, True],
        columns=[False, True],
        fill_value=0
    )

    test = mcnemar(table.values, exact=True)

    return table, test.pvalue


# PPA comparison: restrict to autoantibody-positive cases
ppa_table, ppa_p = compare_paired(
    ai_iif_positive[ab_positive],
    human_iif_positive[ab_positive],
    ab_positive[ab_positive]
)

# NPA comparison: restrict to autoantibody-negative cases
npa_table, npa_p = compare_paired(
    ai_iif_positive[~ab_positive],
    human_iif_positive[~ab_positive],
    ab_positive[~ab_positive]
)

print(f"McNemar PPA: p = {ppa_p:.4g}")
print(f"McNemar NPA: p = {npa_p:.4g}")

In [ ]:
# Figure 3: pattern–autoantibody association matrix

ab_binary = df[AUTOANTIBODIES].ge(AB_POS_THRESHOLD)

fig3_matrix = pd.DataFrame(
    index=FIG3_PATTERN_MAP,
    columns=AUTOANTIBODIES,
    dtype=float,
)
fig3_annot = pd.DataFrame(
    "", index=FIG3_PATTERN_MAP, columns=AUTOANTIBODIES, dtype=object
)

for row_label, col in FIG3_PATTERN_MAP.items():
    if "(AI)" in row_label:
        pattern_pos = ai_positive(df[col])
    else:
        pattern_pos = human_positive(df[col])

    for ab in AUTOANTIBODIES:
        disease = ab_binary[ab]
        denom = int(disease.sum())
        num = int((pattern_pos & disease).sum())

        if denom == 0:
            fig3_matrix.loc[row_label, ab] = np.nan
            continue

        pct = num / denom * 100
        fig3_matrix.loc[row_label, ab] = pct
        fig3_annot.loc[row_label, ab] = f"{num}/{denom}\n({pct:.1f}%)"

fig, ax = plt.subplots(figsize=(19, 10))
sns.heatmap(
    fig3_matrix,
    cmap=cmap_green,
    vmin=0, vmax=100,
    annot=fig3_annot, fmt="",
    linewidths=0.5, linecolor="white",
    cbar=False,
    annot_kws={"size": 6.5, "color": "black"},
    ax=ax,
)

ax.set_xlabel("Specific autoantibody", fontweight="bold")
ax.set_ylabel(
    "HEp-2 IIF interpretation (AI-suggested vs. human-reviewed)",
    fontweight="bold",
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.show()

# Result 4. AI probability distributions and discrimination for autoantibody positivity

This section generates:

- **Figure 4:** AI probability distributions according to corresponding autoantibody status.
- **Supplementary Table S3:** median (IQR) AI probabilities and Mann–Whitney U tests.
- **Table 2:** positive percentage agreement (PPA) and negative percentage agreement (NPA) at the manufacturer-defined 50% probability threshold, together with the c-statistic and 95% CI for discrimination.

Agreement at the manufacturer-defined **50% AI probability threshold** is quantified using PPA and NPA.  
Continuous probability analyses use only cases with an available AI probability, with discrimination for corresponding autoantibody positivity summarized using the **c-statistic**.

In [ ]:
# Figure 4 + Supplementary Table S3

pairs = [
    (pattern, info["ai_col"], ab)
    for pattern, info in AI_AB_MAP.items()
    for ab in info["antibodies"]
]

summary_rows = []

n_plots = len(pairs)
n_cols = 4
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(16, 3.4 * n_rows),
    sharey=True,
)
axes = np.atleast_1d(axes).ravel()

palette = {"Negative": LIGHT_GREEN, "Positive": MAIN_GREEN}
group_order = ["Negative", "Positive"]

for i, (pattern, ai_col, ab) in enumerate(pairs):
    ax = axes[i]

    plot_df = df[[ai_col, ab]].copy()
    plot_df[ai_col] = numeric_series(plot_df[ai_col])
    plot_df[ab] = numeric_series(plot_df[ab])
    plot_df = plot_df.dropna()

    plot_df["Status"] = np.where(
        plot_df[ab].ge(AB_POS_THRESHOLD),
        "Positive",
        "Negative",
    )

    neg = plot_df.loc[plot_df["Status"].eq("Negative"), ai_col]
    pos = plot_df.loc[plot_df["Status"].eq("Positive"), ai_col]

    if len(neg) and len(pos):
        u_stat, p_value = mannwhitneyu(neg, pos, alternative="two-sided")
    else:
        u_stat, p_value = np.nan, np.nan

    summary_rows.append({
        "AI-suggested pattern": pattern,
        "Autoantibody": AB_DISPLAY.get(ab, ab),
        "Positive n": len(pos),
        "Positive median": pos.median() if len(pos) else np.nan,
        "Positive Q1": pos.quantile(0.25) if len(pos) else np.nan,
        "Positive Q3": pos.quantile(0.75) if len(pos) else np.nan,
        "Negative n": len(neg),
        "Negative median": neg.median() if len(neg) else np.nan,
        "Negative Q1": neg.quantile(0.25) if len(neg) else np.nan,
        "Negative Q3": neg.quantile(0.75) if len(neg) else np.nan,
        "Mann–Whitney U": u_stat,
        "p-value": p_value,
    })

    sns.violinplot(
        data=plot_df,
        x="Status", y=ai_col,
        hue="Status",
        order=group_order,
        palette=palette,
        inner="box",
        cut=0,
        linewidth=0.8,
        legend=False,
        ax=ax,
    )
    sns.stripplot(
        data=plot_df,
        x="Status", y=ai_col,
        order=group_order,
        color="black",
        size=1.5,
        alpha=0.25,
        jitter=0.18,
        ax=ax,
    )

    if len(neg) and len(pos):
        p_text = "p < 0.001" if p_value < 0.001 else f"p = {p_value:.3f}"
        y_min = plot_df[ai_col].min()
        y_max = plot_df[ai_col].max()
        y_range = max(y_max - y_min, 1)

        y_bracket = y_max + 0.08 * y_range
        ax.plot(
            [0, 0, 1, 1],
            [y_bracket, y_bracket + 0.02*y_range,
             y_bracket + 0.02*y_range, y_bracket],
            color="black", linewidth=0.8,
        )
        ax.text(
            0.5, y_max + 0.13*y_range,
            p_text,
            ha="center", va="bottom",
            fontsize=8,
            fontweight="bold" if p_value < 0.05 else "normal",
        )
        ax.set_ylim(y_min - 0.05*y_range, y_max + 0.25*y_range)

    ax.set_title(f"{pattern}\n{ab}", fontweight="bold", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("AI probability" if i % n_cols == 0 else "")

for j in range(n_plots, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

supp_s3_raw = pd.DataFrame(summary_rows)

# Result 5. AI-guided anti-DFS70 testing strategy

This section reproduces the receiver operating characteristic analysis of dense fine speckled (AC-2) AI probability for anti-DFS70 positivity (≥2+), with discrimination summarized using the **c-statistic**. This analysis supports evaluation of AI probability thresholds for downstream anti-DFS70 testing.

In [ ]:
# Figure 5A: DFS probability ROC for anti-DFS70 positivity

dfs_roc = df[["AC_2_P_AI", "DFS70"]].copy()
dfs_roc["AC_2_P_AI"] = numeric_series(dfs_roc["AC_2_P_AI"])
dfs_roc["DFS70"] = numeric_series(dfs_roc["DFS70"])
dfs_roc = dfs_roc.dropna()

y_true = dfs_roc["DFS70"].ge(AB_POS_THRESHOLD).astype(int)
y_score = dfs_roc["AC_2_P_AI"]

fpr, tpr, roc_thresholds = roc_curve(y_true, y_score)

c_statistic = auc(fpr, tpr)
c_low, c_high = bootstrap_auc_ci(y_true, y_score)

fig, ax = plt.subplots(figsize=(7.2, 6.2))

ax.step(fpr, tpr, where="post", color=DARK_GREEN, linewidth=2.5)
ax.fill_between(fpr, tpr, 0, step="post", color=MAIN_GREEN, alpha=0.20)
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)

ax.set_xlabel("False Positive Rate", fontweight="bold")
ax.set_ylabel("True Positive Rate", fontweight="bold")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

ax.legend(
    [f"c-statistic = {c_statistic:.2f} "
     f"(95% CI, {c_low:.2f}–{c_high:.2f})"],
    loc="lower right",
    frameon=False,
)

plt.tight_layout()
plt.show()